# Select a Worker and Tool Environment

A runner owns the worker that performs preprocess, execute and postprocess. A command operation may separately declare its tool Environment. That declaration selects command software; it does not select a runner.

This local example uses installed Artisan definitions and observes the current interpreter. `LocalRunner()` does not install software and its observed mode does not enable persistent cache reuse. The caching tutorials prepare a canonical worker first.

In [ ]:
from __future__ import annotations

from artisan.operations.examples import DataGenerator, DataTransformer, MetricCalculator
from artisan.orchestration import PipelineManager
from artisan.orchestration.runners import LocalRunner
from artisan.utils import tutorial_setup
from artisan.visualization import inspect_pipeline, inspect_step

In [ ]:
from functools import partial
from tempfile import TemporaryDirectory

# Keep this tutorial's stores and source files in one owned temporary workspace.
workspace = TemporaryDirectory(prefix="artisan-tutorial-")
tutorial_setup = partial(tutorial_setup, base_dir=workspace.name)
env = tutorial_setup("worker_selection")
runner = LocalRunner()

In [ ]:
with PipelineManager.create(
    name="worker_selection",
    delta_root=env.delta_root,
    staging_root=env.staging_root,
    working_root=env.working_root,
    default_step_runner=runner,
) as pipeline:
    generated = pipeline.run(
        DataGenerator, name="generate", params={"count": 4, "seed": 42}
    )
    transformed = pipeline.run(
        DataTransformer,
        name="transform",
        inputs={"dataset": pipeline.output("generate", "datasets")},
        params={"scale_factor": 0.5, "variants": 2, "seed": 100},
        batch_strategy={"artifacts_per_unit": 2, "max_workers": 2},
    )
    scored = pipeline.run(
        MetricCalculator,
        name="metrics",
        inputs={"dataset": pipeline.output("transform", "dataset")},
        step_runner=runner,
    )
    summary = pipeline.finalize()
assert summary["overall_success"], summary
run_id = pipeline.config.pipeline_run_id
assert (
    inspect_step(env.delta_root, generated.step_number, pipeline_run_id=run_id).height
    == 4
)
assert (
    inspect_step(env.delta_root, transformed.step_number, pipeline_run_id=run_id).height
    == 8
)
assert (
    inspect_step(env.delta_root, scored.step_number, pipeline_run_id=run_id).height == 8
)
inspect_pipeline(env.delta_root, pipeline_run_id=run_id)

`artifacts_per_unit` partitions scientific work. `max_workers` limits concurrent worker owners; it does not change the operation's output cardinality. The explicit runner on the last call overrides the pipeline default for that submission.

For a prepared worker, construct LocalRunner with an Environment pointing to one canonical Pixi manifest and sibling lock, then call `runner.prepare(Operation)` before running it. A command operation with its own environment also requires that target to be prepared. Operations without a separate tool environment inherit their worker's software.

External providers implement the same whole-worker lifecycle for native and command operations. See [Running Remote Workers](../../how-to-guides/running-remote-workers.md) and the resource-marked Modal tutorials. No endpoint deployment or `compute_provider` parameter is needed.